# 16. Integrated Workflows

Generate three room designs, choose the one with the least surface area, attach its results as metadata, and export it.

**Goal:** Complete one small workflow. Run the cells from top to bottom in a fresh kernel.

**Extra dependencies:** None beyond the base environment.

All examples use metres unless stated otherwise.


In [ ]:
import os
os.environ["TOPOLOGICPY_CORE_BACKEND"] = "pythonocc"

import topologicpy
from topologicpy.Core import Core

assert topologicpy.__version__ == "0.9.86", "Use TopologicPy 0.9.86 for these tutorials."
assert type(Core.Backend()).__name__ == "PythonOCCBackend", "Restart with the pythonocc backend."
print("TopologicPy", topologicpy.__version__, "—", type(Core.Backend()).__name__)


## Generate and compare

This combines parametric construction and measurement. Each room has the same volume; only its proportions change.


In [ ]:
from topologicpy.Cell import Cell
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary

best_room = None
best_area = float("inf")
best_width = None
for width in [3, 4, 6]:
    room = Cell.Prism(width=width, length=24 / width, height=3)
    area = Cell.SurfaceArea(room)
    print("Width:", width, "Surface area:", area)
    if area < best_area:
        best_room = room
        best_area = area
        best_width = width
assert best_width == 4
assert abs(Cell.Volume(best_room) - 72) < 0.0001


## Record the decision and export

The exported geometry carries the selection criterion and its value, so another workflow can understand why this option was chosen. Widths 4 and 6 tie here; the loop keeps the first best option.


In [ ]:
from pathlib import Path

properties = Dictionary.ByKeysValues(
    ["name", "objective", "surface_area"],
    ["Selected room", "minimum surface area among candidates", best_area]
)
best_room = Topology.SetDictionary(best_room, properties)
output = Path("results")
output.mkdir(exist_ok=True)
path = output / "16_selected_room.json"
path.write_text(Topology.JSONString([best_room]), encoding="utf-8")
restored = Topology.ByJSONString(path.read_text(encoding="utf-8"))[0]
assert Dictionary.ValueAtKey(Topology.Dictionary(restored), "name") == "Selected room"
print("Saved:", path)


## Try it yourself

Add visualisation from tutorial 05, or store an access graph using tutorial 11. Extend one step at a time.
